# SBND RWM True Bucket: Figures

- Tech note figures of the extraction steps: Sec. "True Bucket Extraction from Raw RWM Traces" (Steps 1-6, example segments) and its appendix (Steps 2-4 for every AD config, Steps 5-6 for every segment)
- Reads the outputs of `trace_fits.ipynb` and `true_bucket.ipynb`, the stored traces and the calibration CSVs; writes `OUT_DIR/figures/ad_*.png`

## 1. Setup

- Set the paths in the Setup cell: `STORE`, `CALIB_CSV`, `OUT_DIR` (same as `true_bucket.ipynb`)

In [ ]:
import os
import time
os.environ['TZ'] = 'UTC'
time.tzset()

import numpy as np
import pickle
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcdefaults()

import sys
sys.path.insert(0, '../lib')
import paths
from segments import select_chunks, segment_inputs
from ad_response import SEGMENTS, load_ad_term
from true_bucket import AD_CONFIGS_USED, reflection_models, corrected_histograms, tb_context, load_npz, load_tails
import step_figures as sf

# ===== Set these paths =====
STORE = {1: '/path/to/run1_rwm_traces', 2: '/path/to/run2_rwm_traces'}   # stored raw RWM traces: folders of run_<DAQ run>.npz
CALIB_CSV = {1: '/path/to/run1_calibration.csv', 2: '/path/to/run2_calibration.csv'}   # calibration CSVs from SBND_RWM_CALIBRATION
OUT_DIR = '../output'   # output folder, the same in every notebook
paths.set_paths(store=STORE, calib=CALIB_CSV, out=OUT_DIR)

F_TERM = paths.out_file('ad_term.npz')
F_TAILS = paths.out_file('end_of_spill.npz')
F_WB = paths.out_file('corrected_histograms.npz')
F_FITS = paths.out_file('true_bucket_fits.pkl')
FIG_DIR = paths.out_file('figures')
os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update(sf.TN_STYLE)

## 2. Inputs

- Segments, end-of-spill probes and tails, corrected per-bucket histograms, H, taps and group fits from the `true_bucket.ipynb` outputs (no refit)
- One pass over the stored traces (`corrected_histograms`, raw positions, `keep_sub`): per-bucket histograms before Step 2 (corrected + removed part) and the corrected end-of-spill tail

In [ ]:
CHUNKS = {**select_chunks(1), **select_chunks(2)}
DATA = segment_inputs(CHUNKS)
TERM = load_ad_term(F_TERM)
TAILS = load_tails(F_TAILS)
RMOD = reflection_models(TERM, TAILS, list(AD_CONFIGS_USED))
WB = load_npz(F_WB)
with open(F_FITS, 'rb') as f_:
    z_ = pickle.load(f_)
CTX, FIT = tb_context(DATA, WB, fitted=z_), z_['FIT']
WR, TLC = corrected_histograms(RMOD, positions='raw', keep_sub=True)
W = {k: dict(WR[k], S=WR[k]['S'] + WR[k]['S_sub']) for k in SEGMENTS}
for pn in AD_CONFIGS_USED:
    TAILS[pn].update(yc=TLC[pn]['y'], nc=TLC[pn]['n'])
del WR
PKEYS = {pn: [k for k in SEGMENTS if str(CTX['DSF'][k]['grp_config'][0]) == pn] for pn in AD_CONFIGS_USED}
I = dict(DATA=DATA, TERM=TERM, TAILS=TAILS, RMOD=RMOD, W=W, WB=WB, CTX=CTX, FIT=FIT, PKEYS=PKEYS)
{pn: [sf.title(k) for k in ks] for pn, ks in PKEYS.items()}

## 3. Extraction Steps

- AD configs 1.1 (50 Ω) and 2.1 (1 MΩ); example segments Run 1 Before Rotation A and Run 2 Wide

In [ ]:
EXAMPLES = [(1, 'before_a'), (2, 'wide')]
sf.fig_step1(CHUNKS[EXAMPLES[0]]['ad'], STORE[1], FIG_DIR)
sf.fig_step2(I, ('1.1', '2.1'), EXAMPLES, FIG_DIR, 'ad_step2_reflections.png')
sf.fig_step3(I, ('1.1', '2.1'), EXAMPLES, FIG_DIR, 'ad_step3_groups.png')
sf.fig_step4_parts(I, FIG_DIR)
sf.fig_step4_buildup(I, ('1.1', '2.1'), EXAMPLES, FIG_DIR, 'ad_step4_buildup.png')
sf.fig_step5_illustration(I, EXAMPLES[0], FIG_DIR)
sf.fig_step5(I, EXAMPLES, FIG_DIR, 'ad_step5_fit.png')
sf.fig_step6(I, EXAMPLES, FIG_DIR, 'ad_step6_mixture.png')

## 4. Appendix: Every AD Config and Segment

- Steps 2-4: one figure per run (AD configs 1.1 and 1.3, 2.1 and 2.2); Steps 5-6: one figure per run, all segments

In [ ]:
S2_50 = dict(top_ylim=(-0.14, 0.12), b80=((3.5, 0.115), (10, 0.105)), short_txt=(45, 0.075), rt_txt=(45, -0.075), mid_ylim=(-0.2, 0.12))
S2_1M = dict(top_ylim=(-0.44, 0.19), b80=((1.5, 0.17), (5, 0.14)), short_txt=(45, 0.09), rt_txt=(45, -0.30), mid_ylim=(-0.36, 0.2))
APP = {1: (('1.1', '1.3'), [(1, 'before_a'), (1, 'after')], S2_50),
       2: (('2.1', '2.2'), [(2, 'wide'), (2, 'normal_b')], {n: [S2_1M[n], S2_50[n]] for n in S2_50})}
for run, (pns, exs, s2) in APP.items():
    keys = [k for k in SEGMENTS if k[0] == run]
    sf.fig_step2(I, pns, exs, FIG_DIR, f'ad_app_step2_run{run}.png', **s2)
    sf.fig_step3(I, pns, exs, FIG_DIR, f'ad_app_step3_run{run}.png')
    sf.fig_step4_buildup(I, pns, exs, FIG_DIR, f'ad_app_step4_buildup_run{run}.png')
    sf.fig_step5(I, keys, FIG_DIR, f'ad_app_step5_run{run}.png')
    sf.fig_step6(I, keys, FIG_DIR, f'ad_app_step6_run{run}.png')